# Build Your Own RAG System (Solution Notebook)

Facilitator copy. Run this end-to-end before the event.


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from langchain_community.embeddings import HuggingFaceEmbeddings

from rag_pipeline import get_llm_provider

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

provider = get_llm_provider()
api_key = os.getenv("GEMINI_API_KEY")
assert provider == "gemini", "Set LLM_PROVIDER=gemini in .env for this workshop"
assert api_key and api_key != "your_key_here", "Set GEMINI_API_KEY in .env"

print("Setup OK")
print("LLM provider:", provider)
print("Repo root:", ROOT)


In [ ]:
from rag_pipeline import (
    load_documents,
    split_documents,
    build_vector_store,
    load_vector_store,
    answer_question,
)

DOCS_DIR = ROOT / "sample_docs"
CHROMA_DIR = ROOT / "chroma_db"

documents = load_documents(DOCS_DIR)
chunks = split_documents(documents)
vector_store = build_vector_store(chunks, CHROMA_DIR)

print(f"Loaded {len(documents)} documents")
print(f"Created {len(chunks)} chunks")


In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectors = embeddings.embed_documents(
    [
        "The student passed the examination",
        "The learner succeeded in the test",
        "The banana is yellow",
    ]
)


def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print("exam vs test:", round(cosine_similarity(vectors[0], vectors[1]), 3))
print("exam vs banana:", round(cosine_similarity(vectors[0], vectors[2]), 3))


In [ ]:
question = "What is the primary purpose of government in Nigeria?"
result = answer_question(vector_store, question)

print("Answer:", result["answer"])
print("Sources:", result["sources"])


In [ ]:
from rag_pipeline import get_llm

llm = get_llm()
plain = llm.invoke(question)
print("Without RAG:\n", plain.content)
print("\nWith RAG:\n", result["answer"])


In [ ]:
# Optional reload test
reloaded = load_vector_store(CHROMA_DIR)
print("Reloaded store OK:", reloaded._collection.count())
